# Replicating the SWEC-ETHZ *Short-term* LBP + HD seizure detector
**Burrello, Schindler, Benini & Rahimi, "One-shot Learning for iEEG Seizure Detection Using End-to-end Binary Operations: Local Binary Patterns with Hyperdimensional Computing", IEEE BioCAS 2018**
Data and original code: <http://ieeg-swez.ethz.ch/> (Short-term Dataset and Algorithms)

This notebook repeats the whole replication from the lab PC in one place, steps 1 to 10:

| Step | What it does | Time |
|---|---|---|
| 1 | Settings, machine check | seconds |
| 2 | Project folders, VS Code settings, `.gitignore`, `requirements.txt` | seconds |
| 3 | Install Python packages (PyTorch, SciPy, ...) | 2–10 min (first time) |
| 4 | Git repository (optional) | seconds |
| 5 | Download dataset and original code (optional, skipped if already present) | 0 – a few hours |
| 6 | Verify the dataset (16 patients, 100 seizures) | ~5 min |
| 7 | Install the Python 3 / PyTorch 2 version of the LBP+HD code and self-test it | ~1 min |
| 8 | **Train and test LBP+HD for every patient** (smoke test on patient 6 first) | **~2 days on CPU** (`FOLD_MODE="full"`) |
| 9 | Rebuild `summary.csv` and recompute metrics under the paper's protocol | ~5 min |
| 10 | Compare with the paper's Tables I–II, check against the lab run, plot | seconds |

**How to use it**
* Read `README.txt` first: it explains the one-time setup (Python 3.11 environment and selecting it as the kernel).
* Run the cells from top to bottom. **After any kernel restart, run the Step 1 cells again first.**
* Step 8 is long. It can be stopped and restarted at any time: patients whose predictions are already complete are skipped.

## Step 1 — Settings and machine check
Edit the settings cell only if your folders or choices differ. Everything else is derived from it.

* `FOLD_MODE = "full"` repeats the lab run exactly (sliding k-fold for every patient, 679 seizure tests).
* `FOLD_MODE = "original"` runs only what the paper's tables need (k-fold for Table I patients, a single split for Table II patients, 246 tests, about 2.8× faster). The paper-protocol results in Step 10 are identical in both modes. Only the "released-code" columns differ.

In [ ]:
# ============================ SETTINGS (edit if needed) ============================
import os, sys, shutil, platform, subprocess, time, json
from pathlib import Path

ROOT = Path(os.environ.get("SWEC_ROOT", r"C:\Users\mrahman\sEEG"))   # project root

DOWNLOAD_DATA  = False   # True: download the 19 dataset zips from ETH if patient folders are missing
DOWNLOAD_CODE  = True    # original ETH code (reference, baselines, TBME 2019); LBP+HD itself is embedded below
FOLD_MODE      = "full"  # "full" = same as the lab run;  "original" = paper protocol only (faster)
SEED           = 1       # random seed for the hypervector item memories (authors' value)
RUN_PATIENTS   = [6, 11, 13, 16, 8, 2, 4, 1, 9, 7, 5, 12, 14, 15, 10, 3]   # paper patient numbers, smallest first
RERUN_EXISTING = False   # False: skip patients whose prediction files are already complete
USE_GIT        = True    # record the code in a Git repository (skipped automatically if Git is not installed)
# ===================================================================================

DATA_DIR    = ROOT / "data" / "raw" / "swec_ethz_short"
DATA_ZIPS   = DATA_DIR / "_zips"
CODE_ROOT   = ROOT / "code" / "swec_ethz_short"
LBPHD_DIR   = CODE_ROOT / "LBP+HD" / "LBP+HD"          # where the ported scripts live and run
SCRIPTS_DIR = ROOT / "scripts"
RESULTS     = ROOT / "results"
LBPHD_RES   = RESULTS / "swec_short" / "LBP+HD"
PLOTS       = RESULTS / "plots"

# settings read by swec_config.py (in this kernel and in every subprocess started below)
os.environ["SWEC_ROOT"]  = str(ROOT)
os.environ["SWEC_FOLDS"] = FOLD_MODE
os.environ["SWEC_SEED"]  = str(SEED)
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUNBUFFERED"] = "1"

ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)          # the %%writefile cells below use paths relative to ROOT
for p in (str(LBPHD_DIR), str(SCRIPTS_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)


def run(cmd, cwd=None, log=None, append=False, env=None):
    """Run a command, stream its output here and (optionally) into a log file. Returns the exit code."""
    env = {**os.environ, **(env or {})}
    fh = open(log, "a" if append else "w", encoding="utf-8") if log else None
    try:
        proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding="utf-8", errors="replace", bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            if fh:
                fh.write(line)
                fh.flush()
        return proc.wait()
    finally:
        if fh:
            fh.close()

print("Project root :", ROOT)
print("Data folder  :", DATA_DIR)
print("Fold mode    :", FOLD_MODE, "| seed:", SEED)

In [ ]:
# Machine check
print("Python      :", sys.version.split()[0], "|", sys.executable)
print("OS          :", platform.platform())
print("CPU cores   :", os.cpu_count())
free_gb = shutil.disk_usage(ROOT.anchor or str(ROOT)).free / 1e9
print(f"Free disk   : {free_gb:.0f} GB on {ROOT.anchor}")

if sys.version_info[:2] != (3, 11):
    print("WARNING: tested with Python 3.11. 3.10/3.12 should work; 3.13+ cannot install numpy<2.")
if ".venv" not in sys.executable and "venv" not in sys.prefix.lower():
    print("NOTE: this kernel does not look like a virtual environment. See README.txt, section 2.")
if free_gb < 30:
    print("WARNING: less than 30 GB free. The data needs ~12 GB unpacked (+ ~12 GB if the zips are downloaded).")

nvsmi = shutil.which("nvidia-smi")
if nvsmi:
    subprocess.run([nvsmi, "--query-gpu=name,driver_version,memory.total", "--format=csv"])
else:
    print("GPU         : no NVIDIA driver found -> the code will run on the CPU (slow but works)")
print("Git         :", shutil.which("git") or "not installed (Step 4 will be skipped)")

## Step 2 — Project folders, VS Code settings, `.gitignore`, `requirements.txt`
Existing files are never overwritten.

In [ ]:
for d in (DATA_DIR, DATA_ZIPS, CODE_ROOT, LBPHD_DIR, SCRIPTS_DIR, LBPHD_RES, PLOTS, ROOT / ".vscode"):
    d.mkdir(parents=True, exist_ok=True)

files = {
    ROOT / ".gitignore": "data/\n.venv/\n**/_zips/\n__pycache__/\n*.pyc\nresults/\ndesktop.ini\n.ipynb_checkpoints/\n",
    ROOT / "requirements.txt": (
        "# SWEC-ETHZ short-term replication (Python 3.11). PyTorch is installed separately in Step 3.\n"
        "numpy>=1.26,<2.0\nscipy>=1.11\nh5py>=3.10\nscikit-learn>=1.3\npandas>=2.1\nmatplotlib>=3.8\n"
        "tqdm\nipykernel\nnotebook\n"),
    ROOT / ".vscode" / "settings.json": json.dumps({
        "python.defaultInterpreterPath": "${workspaceFolder}\\.venv\\Scripts\\python.exe",
        "python.terminal.activateEnvironment": True,
        "python.terminal.executeInFileDir": True,
        "files.watcherExclude": {"**/data/**": True, "**/.venv/**": True, "**/results/**": True},
        "search.exclude": {"**/data/**": True, "**/.venv/**": True},
    }, indent=2) + "\n",
    ROOT / ".vscode" / "launch.json": json.dumps({
        "version": "0.2.0",
        "configurations": [{
            "name": "Run current file (own folder)", "type": "debugpy", "request": "launch",
            "program": "${file}", "cwd": "${fileDirname}", "console": "integratedTerminal",
            "env": {"SWEC_PATIENT": "6"},
        }],
    }, indent=2) + "\n",
}
for path, text in files.items():
    if path.exists():
        print("[keep]   ", path.relative_to(ROOT))
    else:
        path.write_text(text, encoding="ascii")
        print("[created]", path.relative_to(ROOT))

## Step 3 — Install the Python packages
Installs into **this notebook's kernel** (your `.venv`). PyTorch is installed only if it is missing: the CUDA build if an NVIDIA driver is present, otherwise the smaller CPU build.

If this cell changed NumPy or PyTorch, **restart the kernel** (VS Code: *Restart* button; Jupyter: *Kernel → Restart*), then run the Step 1 cells again.

In [ ]:
pip = [sys.executable, "-m", "pip"]
run(pip + ["install", "--upgrade", "pip"])
run(pip + ["install", "-r", str(ROOT / "requirements.txt")])

try:
    import torch  # noqa: F401
    print("PyTorch already installed:", torch.__version__)
except ImportError:
    # cu126 = CUDA 12.6 build. If pytorch.org lists a newer CUDA for your driver, change it here.
    index = "https://download.pytorch.org/whl/cu126" if shutil.which("nvidia-smi") else "https://download.pytorch.org/whl/cpu"
    print("Installing PyTorch from", index)
    run(pip + ["install", "torch", "--index-url", index])
    print("\n>>> PyTorch was installed: restart the kernel, then run the Step 1 cells again.")

In [ ]:
import numpy, scipy, sklearn, pandas, matplotlib, h5py, torch
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__,
      "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "running on CPU")
assert numpy.__version__.startswith("1."), "numpy 2.x found: run  pip install \"numpy<2\"  and restart the kernel"

## Step 4 — Git repository (optional)
Records every version of the code, so the changes from the authors' original are documented for the methods chapter. `data/`, `.venv/` and `results/` are excluded by `.gitignore`.
If Git asks for a name and email, set them once in a terminal:
`git config --global user.name "Your Name"` and `git config --global user.email "you@example.com"`.

In [ ]:
GIT = shutil.which("git") if USE_GIT else None

def git_commit(message):
    """Commit everything that changed (no-op if Git is off or nothing changed)."""
    if not GIT:
        return
    run([GIT, "add", "-A"], cwd=ROOT)
    if subprocess.run([GIT, "diff", "--cached", "--quiet"], cwd=ROOT).returncode == 0:
        print("[git] nothing new to commit")
        return
    if run([GIT, "commit", "-q", "-m", message], cwd=ROOT) == 0:
        print("[git] committed:", message)
    else:
        print("[git] commit failed - set user.name / user.email (see above) and run this cell again")

if not GIT:
    print("Git not used (USE_GIT=False or Git not installed) - continuing without it.")
elif not (ROOT / ".git").exists():
    run([GIT, "init"], cwd=ROOT)
    git_commit("Project skeleton (.gitignore, requirements, VS Code settings)")
else:
    print("Git repository already exists:", ROOT / ".git")

## Step 5 — Download the dataset and the original code (optional)
* **Dataset:** if `data\raw\swec_ethz_short\ID1 … ID16` already contain the `Sz*.mat` files (copied from the lab PC, or downloaded by hand), nothing is downloaded. Otherwise set `DOWNLOAD_DATA = True` in Step 1. That downloads 19 zips (each < 1 GB) and unpacks them. ID4a/4b, ID13a/b and ID14a/b are merged into ID4, ID13 and ID14. Interrupted downloads resume; finished ones are skipped.
* **Original code** (`DOWNLOAD_CODE = True`): the authors' LBP+HD, baselines and TBME 2019 packages, kept for reference. The runnable Python 3 version of LBP+HD is embedded in Step 7, so this download is not required.

In [ ]:
import urllib.request, zipfile

DATA_BASE = "https://iis-people.ee.ethz.ch/~ieeg/BioCAS2018/dataset"
DATA_IDS = ["ID1", "ID2", "ID3", "ID4a", "ID4b", "ID5", "ID6", "ID7", "ID8", "ID9", "ID10",
            "ID11", "ID12", "ID13a", "ID13b", "ID14a", "ID14b", "ID15", "ID16"]
CODE_URLS = {
    "LBP+HD":                "https://iis-people.ee.ethz.ch/~ieeg/BioCAS2018/codes/LBP+HD.zip",
    "baselines":             "https://iis-people.ee.ethz.ch/~ieeg/BioCAS2018/codes/baselines.zip",
    "LBP+HD_Identification": "http://ieeg-swez.ethz.ch/TBME2019/LBP+HD_Identification.zip",
}

def zip_ok(path):
    try:
        with zipfile.ZipFile(path) as z:
            return len(z.namelist()) > 0
    except Exception:
        return False

def download(url, out, attempts=5):
    """Download with resume (HTTP Range) and retries; skips a file that is already a valid zip."""
    out = Path(out)
    if out.exists() and zip_ok(out):
        print(f"  [skip] {out.name}")
        return True
    for a in range(1, attempts + 1):
        have = out.stat().st_size if out.exists() else 0
        req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                mode = "ab" if (have and r.status == 206) else "wb"
                total = int(r.headers.get("Content-Length") or 0) + (have if mode == "ab" else 0)
                done = have if mode == "ab" else 0
                with open(out, mode) as f:
                    while True:
                        chunk = r.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk)
                        done += len(chunk)
                        if total:
                            print(f"\r  [get ] {out.name}: {done/1e6:,.0f} / {total/1e6:,.0f} MB", end="")
            print()
            if zip_ok(out):
                return True
            out.unlink(missing_ok=True)          # corrupt -> start clean next time
        except Exception as e:
            print(f"\n  attempt {a} failed for {out.name}: {e}")
            if getattr(e, "code", None) == 416:   # range not satisfiable -> start clean
                out.unlink(missing_ok=True)
        time.sleep(10)
    print(f"  [FAIL] {url}")
    return False

def unpack_patient_zip(zpath, tag):
    """Move every file of the zip into DATA_DIR/ID<n> (ID4a + ID4b -> ID4)."""
    patient = tag.rstrip("ab")
    dest = DATA_DIR / patient
    marker = dest / f".unpacked_{tag}"
    if marker.exists():
        return
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            name = Path(info.filename).name
            if info.is_dir() or "__MACOSX" in info.filename or name.startswith("._"):
                continue
            target = dest / name
            if target.exists():
                target = dest / f"{Path(name).stem}_{tag}{Path(name).suffix}"
                print("  WARNING name clash ->", target.name)
            with z.open(info) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, 1 << 20)
    marker.touch()
    print(f"  [unzip] {tag} -> {dest}")

def mat_count(pid):
    d = DATA_DIR / pid
    return len(list(d.glob("Sz*.mat"))) if d.is_dir() else 0

# ---- original code (small) ----
if DOWNLOAD_CODE:
    print("== Original source code ==")
    (CODE_ROOT / "_zips").mkdir(parents=True, exist_ok=True)
    for name, url in CODE_URLS.items():
        z = CODE_ROOT / "_zips" / f"{name}.zip"
        dest = CODE_ROOT / name
        if download(url, z) and not dest.exists():
            with zipfile.ZipFile(z) as zz:
                zz.extractall(dest)
            print(f"  [unzip] {name} -> {dest}")

# ---- dataset ----
present = {f"ID{i}": mat_count(f"ID{i}") for i in range(1, 17)}
missing = [p for p, n in present.items() if n == 0]
print("\n== Dataset ==")
print("Seizure files already present:", sum(present.values()), "| patients missing:", missing or "none")
if missing and DOWNLOAD_DATA:
    for tag in DATA_IDS:
        if tag.rstrip("ab") not in missing:
            continue
        z = DATA_ZIPS / f"{tag}.zip"
        if download(f"{DATA_BASE}/{tag}.zip", z):
            unpack_patient_zip(z, tag)
elif missing:
    print("-> Copy the ID folders into", DATA_DIR, "or set DOWNLOAD_DATA = True in Step 1 and run this cell again.")

## Step 6 — Verify the dataset
Writes `scripts\verify_swec_short.py` (the same checker used on the lab PC), then checks every recording:
* **Counts:** 16 patients and 100 seizure files.
* **Recording layout:** 512 Hz, with 3 min before and 3 min after each seizure.
* **Seizure length:** 10–1002 s.
* **Outputs:** `results\swec_short_inventory.csv` and an example plot.

In [ ]:
os.chdir(ROOT); SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile scripts/verify_swec_short.py
#!/usr/bin/env python
"""
verify_swec_short.py
--------------------
Sanity-check the SWEC-ETHZ Short-term iEEG dataset after download, and provide
a small loader you can import from your own notebooks / deep-learning code.

Dataset facts (http://ieeg-swez.ethz.ch/):
  * 16 patients, 100 seizures, 512 Hz, band-pass 0.5-150 Hz
  * one .mat file per seizure (Sz1.mat, Sz2.mat, ...) holding variable EEG, shape T x M
    (T samples, M electrodes)
  * every recording = 3 min pre-ictal + ictal (10-1002 s) + 3 min post-ictal,
    so the seizure starts at sample 92160 and ends at sample T - 92160

Usage (from C:\\Users\\mrahman\\sEEG with the .venv active):
  python scripts/verify_swec_short.py                 # inventory of all patients -> results/
  python scripts/verify_swec_short.py --plot ID1 1    # also plot ID1/Sz1.mat to a PNG

In your own code:
  import sys; sys.path.append(r"C:\\Users\\mrahman\\sEEG\\scripts")
  from verify_swec_short import load_seizure
  rec = load_seizure("ID1", 1)   # rec["eeg"] is channels x samples, float32
"""
from __future__ import annotations

import argparse
import csv
import re
import sys
from pathlib import Path

import numpy as np

FS = 512                      # Hz
EDGE = 3 * 60 * FS            # 92160 samples of pre-/post-ictal context
DEFAULT_ROOT = Path(r"C:\Users\mrahman\sEEG")
DATA_SUBDIR = Path("data") / "raw" / "swec_ethz_short"


def _natural_key(p: Path):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.stem)]


def _pid(patient) -> str:
    s = str(patient)
    return f"ID{s}" if s.isdigit() else s


def read_eeg(mat_path) -> np.ndarray:
    """Return the EEG matrix as stored in the file (documented as T samples x M channels)."""
    from scipy.io import loadmat
    try:
        return np.asarray(loadmat(str(mat_path), variable_names=["EEG"])["EEG"])
    except NotImplementedError:          # MATLAB v7.3 (HDF5) file
        import h5py
        with h5py.File(mat_path, "r") as f:
            return np.asarray(f["EEG"]).T  # h5py returns MATLAB arrays transposed


def load_seizure(patient, seizure: int, root: Path | str = DEFAULT_ROOT) -> dict:
    """Load one seizure recording.

    Returns dict with:
      eeg            channels x samples, float32
      fs             512
      onset_sample   92160
      offset_sample  n_samples - 92160
      path           file path
    """
    path = Path(root) / DATA_SUBDIR / _pid(patient) / f"Sz{seizure}.mat"
    eeg = read_eeg(path)
    if eeg.shape[0] < eeg.shape[1]:      # recordings are >=370 s long, so samples >> channels
        eeg = eeg.T
    n_samples = eeg.shape[0]
    return {
        "eeg": np.ascontiguousarray(eeg.T, dtype=np.float32),
        "fs": FS,
        "onset_sample": EDGE,
        "offset_sample": n_samples - EDGE,
        "path": path,
    }


def inventory(data_dir: Path):
    rows, problems = [], []
    patients = sorted(
        (d for d in data_dir.iterdir() if d.is_dir() and re.fullmatch(r"ID\d+", d.name)),
        key=lambda d: int(d.name[2:]),
    )
    if not patients:
        problems.append(f"No ID* folders found in {data_dir}")
    for pdir in patients:
        files = sorted(pdir.glob("*.mat"), key=_natural_key)
        if not files:
            problems.append(f"{pdir.name}: no .mat files")
        for f in files:
            tag = f"{pdir.name}/{f.name}"
            try:
                eeg = read_eeg(f)
            except Exception as e:                       # noqa: BLE001
                problems.append(f"{tag}: cannot read ({type(e).__name__}: {e})")
                continue
            if eeg.ndim != 2:
                problems.append(f"{tag}: EEG is not 2-D, shape={eeg.shape}")
                continue
            T, M = eeg.shape
            if T < M:
                problems.append(f"{tag}: stored as {eeg.shape} (channels x samples?) - transposed for checks")
                T, M = M, T
            ictal_s = (T - 2 * EDGE) / FS
            n_nan = int(np.isnan(eeg).sum()) if np.issubdtype(eeg.dtype, np.floating) else 0
            if ictal_s <= 0:
                problems.append(f"{tag}: shorter than 6 min, onset/offset rule does not apply")
            elif not (9.5 <= ictal_s <= 1002.5):
                problems.append(f"{tag}: ictal duration {ictal_s:.1f} s outside documented 10-1002 s")
            if n_nan:
                problems.append(f"{tag}: {n_nan} NaN values")
            rows.append({
                "patient": pdir.name, "file": f.name, "n_samples": T, "n_channels": M,
                "duration_s": round(T / FS, 1), "ictal_s": round(ictal_s, 1),
                "dtype": str(eeg.dtype), "n_nan": n_nan,
                "size_mb": round(f.stat().st_size / 1e6, 1),
            })
            print(f"  {tag:<16} {M:>4} ch  {T / FS:>7.1f} s total  {ictal_s:>7.1f} s ictal")
            del eeg
    return rows, problems


def plot_seizure(patient, seizure: int, root: Path, out_dir: Path, n_channels: int = 8) -> Path:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    rec = load_seizure(patient, seizure, root)
    x = rec["eeg"][:n_channels]
    t = np.arange(x.shape[1]) / FS
    spacing = float(5 * np.nanstd(x)) or 1.0
    fig, ax = plt.subplots(figsize=(14, 0.6 * len(x) + 2))
    for i, ch in enumerate(x):
        ax.plot(t, ch - i * spacing, lw=0.4, color="k")
    for s, label in [(rec["onset_sample"], "onset"), (rec["offset_sample"], "offset")]:
        ax.axvline(s / FS, color="r", ls="--", lw=1)
        ax.text(s / FS, spacing, f" {label}", color="r", va="bottom")
    ax.set_yticks([-i * spacing for i in range(len(x))])
    ax.set_yticklabels([f"ch{i + 1}" for i in range(len(x))])
    ax.set_xlabel("time (s)")
    ax.set_title(f"{_pid(patient)} Sz{seizure} - first {len(x)} of {rec['eeg'].shape[0]} channels")
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{_pid(patient)}_Sz{seizure}.png"
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return out


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--root", type=Path, default=DEFAULT_ROOT, help="project root (default: %(default)s)")
    ap.add_argument("--out", type=Path, default=None, help="CSV path (default: <root>/results/swec_short_inventory.csv)")
    ap.add_argument("--plot", nargs=2, metavar=("PATIENT", "SZ"), help="also plot one seizure, e.g. --plot ID1 1")
    args = ap.parse_args()

    data_dir = args.root / DATA_SUBDIR
    out_csv = args.out or (args.root / "results" / "swec_short_inventory.csv")
    print(f"Scanning {data_dir}")
    rows, problems = inventory(data_dir)

    if rows:
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        with open(out_csv, "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)

    print("\nPer patient:")
    for pid in sorted({r["patient"] for r in rows}, key=lambda s: int(s[2:])):
        rs = [r for r in rows if r["patient"] == pid]
        chans = sorted({r["n_channels"] for r in rs})
        print(f"  {pid:<5} {len(rs):>3} seizures  channels={chans}  ictal total={sum(r['ictal_s'] for r in rs):.0f} s")

    n_pat = len({r["patient"] for r in rows})
    print(f"\nPatients: {n_pat} (expected 16)   Seizure files: {len(rows)} (expected 100)")
    if rows:
        print(f"Inventory written to {out_csv}")
    if problems:
        print("\nProblems found:")
        for p in problems:
            print("  -", p)
    else:
        print("No problems found.")

    if args.plot:
        png = plot_seizure(args.plot[0], int(args.plot[1]), args.root, args.root / "results" / "plots")
        print(f"Plot saved to {png}")

    ok = n_pat == 16 and len(rows) == 100 and not problems
    return 0 if ok else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
import importlib, csv
import verify_swec_short as vss
importlib.reload(vss)

rows, problems = vss.inventory(DATA_DIR)
inv_csv = RESULTS / "swec_short_inventory.csv"
if rows:
    with open(inv_csv, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
n_pat = len({r["patient"] for r in rows})
print(f"\nPatients: {n_pat} (expected 16)   Seizure files: {len(rows)} (expected 100)   -> {inv_csv}")
print("Problems:", *problems, sep="\n  - ") if problems else print("No problems found.")
assert n_pat == 16 and len(rows) == 100, "Dataset incomplete - fix Step 5 before continuing."

In [ ]:
from IPython.display import Image, display
png = vss.plot_seizure("ID1", 1, ROOT, PLOTS)      # red lines = expert-marked onset (180 s) and end
display(Image(filename=str(png)))

## Step 7 — Install the Python 3 / PyTorch 2 version of LBP+HD
The authors' code was written for Python 2.7 and an old PyTorch on a GPU server. The four files below are the ported versions that produced the lab results; they are byte-for-byte the files from the lab PC. A fifth file, `paper_protocol_metrics.py`, computes the paper-protocol table.

Changes from the original, each marked `PORT:` in the code (report these in the methods chapter):
1. **File names and folds made consistent.** The released training and post-processing scripts used different file names and different fold counts, so they could not run together.
2. **Training-time bug fixed.** From the second fold onward, the original training loaded one seizure but used another seizure's start and end times. `SWEC_ORIGINAL_TRAIN_INDEX=1` restores the original behaviour.
3. **Patient numbering resolved.** The paper's patient numbers map to the dataset folders through `Patient_trasformation` (for example, paper patient 6 = folder ID7). The mapping is detected automatically from the seizure counts.
4. **Python 3 and PyTorch 2 compatibility.** Runs on CPU or GPU. The faster encoder gives bit-identical output to the original algorithm; `SWEC_FAST=0` uses the original instead.

In [ ]:
os.chdir(ROOT); LBPHD_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile code/swec_ethz_short/LBP+HD/LBP+HD/swec_config.py
#!/usr/bin/env python
"""
swec_config.py  (NEW file added for the Python 3 / PyTorch 2 port)

Settings shared by iEEGClassifierLBP+HD.py and HD_PostProcessing.py, so the two
scripts always agree on paths, folds, patient numbers and device.

All numeric tables are copied verbatim from the original ETH Zurich scripts.

Environment variables (set in PowerShell with  $env:NAME = "value"):
  SWEC_PATIENT   patient number(s) as used in the scripts: "6", "1,2,3" or "all"
  SWEC_FOLDS     "full" (default) = sliding-window CV, K = seizures - train + 1
                 "original"       = the shorter K list hard-coded in the classifier
                 "<n>"            = only the first n folds (quick test)
  SWEC_GPU       GPU index (default 0). CPU is used if CUDA is unavailable.
  SWEC_SEED      torch.manual_seed for the random item memories (default 1, as original)
  SWEC_ID_MAP    force the patient -> data-folder mapping: direct | transform | inverse
  SWEC_ROOT      project root (default C:\\Users\\mrahman\\sEEG)
"""
import os
from pathlib import Path

import torch

# ---------------------------------------------------------------- paths
PROJECT_ROOT = Path(os.environ.get("SWEC_ROOT", r"C:\Users\mrahman\sEEG"))
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "swec_ethz_short"          # ID1..ID16\Sz*.mat
RESULTS_DIR = PROJECT_ROOT / "results" / "swec_short" / "LBP+HD"
PRED_DIR = RESULTS_DIR / "predictions"

FS = 512

# ---------------------------------------------------------------- tables (verbatim)
# number of seizures of each patient
seizureN = [5, 4, 14, 4, 6, 2, 7, 3, 7, 13, 2, 10, 2, 10, 9, 2]
# number of seizures used for training (iEEGClassifierLBP+HD.py; patient 9 = 4,
# "the third seizure is an artifact!"). HD_PostProcessing.py had 3 for patient 9 but never used it.
Seizure_train = [2, 1, 3, 1, 1, 1, 3, 1, 4, 3, 1, 6, 1, 4, 2, 1]
# folds hard-coded in HD_PostProcessing.py  (== seizureN - Seizure_train + 1 for every patient)
crossValidationK_full = [4, 4, 12, 4, 6, 2, 5, 3, 4, 11, 2, 5, 2, 7, 8, 2]
# folds hard-coded in iEEGClassifierLBP+HD.py (1 fold for patients 3, 7, 9, 10, 12, 14)
crossValidationK_original = [4, 4, 1, 4, 6, 2, 1, 3, 1, 1, 2, 1, 2, 1, 8, 2]
# post-processing threshold of each patient (HD_PostProcessing.py)
threshold = [-9.5/10, -9.5/10, -7.5/10, -9.5/10, -9.5/10, -8.5/10, -9.5/10, -7.5/10,
             -8.5/10, -8.5/10, -9.5/10, -6.5/10, -8.5/10, -5.5/10, -9.5/10, -8.5/10]
# "inserted to match the ID of the anonymous dataset uploaded on the webpage, with the ID of the paper"
Patient_trasformation = [10, 6, 4, 2, 16, 7, 14, 15, 13, 1, 8, 5, 3, 12, 9, 11]
_inverse = [0] * 16
for _k, _v in enumerate(Patient_trasformation):
    _inverse[_v - 1] = _k + 1

assert all(n - t + 1 == k for n, t, k in zip(seizureN, Seizure_train, crossValidationK_full))

# ---------------------------------------------------------------- run settings
def get_device() -> torch.device:
    gpu = int(os.environ.get("SWEC_GPU", "0"))
    if torch.cuda.is_available():
        return torch.device(f"cuda:{gpu}")
    return torch.device("cpu")


def seed() -> int:
    return int(os.environ.get("SWEC_SEED", "1"))


def patients_from_env(default):
    raw = os.environ.get("SWEC_PATIENT", "").strip().lower()
    if not raw:
        print(f"[config] SWEC_PATIENT not set - using default {default}")
        return list(default)
    if raw == "all":
        return list(range(1, 17))
    pats = [int(p) for p in raw.replace(" ", "").split(",") if p]
    for p in pats:
        if not 1 <= p <= 16:
            raise ValueError(f"SWEC_PATIENT must be 1..16, got {p}")
    return pats


def folds(patient: int) -> int:
    mode = os.environ.get("SWEC_FOLDS", "full").strip().lower()
    full = crossValidationK_full[patient - 1]
    if mode == "full":
        return full
    if mode == "original":
        return crossValidationK_original[patient - 1]
    return max(1, min(int(mode), full))


# ---------------------------------------------------------------- data folders
_MAPS = {
    "direct": lambda p: p,                                   # script patient p -> ID{p}
    "transform": lambda p: Patient_trasformation[p - 1],     # p -> ID{Patient_trasformation[p-1]}
    "inverse": lambda p: _inverse[p - 1],                    # the reverse direction
}
_mapping_cache = None


def _count_seizures(folder_id: int):
    d = DATA_DIR / f"ID{folder_id}"
    if not d.is_dir():
        return None
    return sum(1 for _ in d.glob("Sz*.mat"))


def mapping() -> str:
    """Pick the patient-number -> data-folder mapping whose seizure counts match seizureN.

    The original scripts are inconsistent (Pat{Patient} in some lines,
    Pat{Patient_trasformation[Patient-1]} in others), so we test the downloaded data.
    """
    global _mapping_cache
    if _mapping_cache:
        return _mapping_cache
    forced = os.environ.get("SWEC_ID_MAP", "").strip().lower()
    if forced:
        if forced not in _MAPS:
            raise ValueError(f"SWEC_ID_MAP must be one of {list(_MAPS)}")
        _mapping_cache = forced
        print(f"[config] patient->folder mapping forced to '{forced}'")
        return forced
    scores = {}
    for name, f in _MAPS.items():
        ok = bad = 0
        for p in range(1, 17):
            c = _count_seizures(f(p))
            if c is None:
                continue
            ok, bad = (ok + 1, bad) if c == seizureN[p - 1] else (ok, bad + 1)
        scores[name] = (ok, bad)
    good = [n for n, (ok, bad) in scores.items() if bad == 0 and ok > 0]
    if not good:
        lines = [f"  {n:<9} matching patients={ok}, mismatching={bad}" for n, (ok, bad) in scores.items()]
        raise RuntimeError(
            "Seizure-file counts in " + str(DATA_DIR) + " do not match the paper's table "
            "under any patient mapping:\n" + "\n".join(lines) +
            "\nRun scripts\\verify_swec_short.py and check the downloads.")
    best = max(good, key=lambda n: scores[n][0])
    ties = [n for n in good if scores[n][0] == scores[best][0]]
    if len(ties) > 1:
        best = ties[0]
        print(f"[config] mappings {ties} fit equally well (download more patients to disambiguate); using '{best}'")
    _mapping_cache = best
    print(f"[config] patient->folder mapping: '{best}' ({scores[best][0]} patients checked)")
    return best


def data_folder(patient: int) -> Path:
    return DATA_DIR / f"ID{_MAPS[mapping()](patient)}"


def seizure_file(patient: int, seizure: int) -> Path:
    path = data_folder(patient) / f"Sz{seizure}.mat"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def prediction_file(patient: int, seizure: int, cross: int) -> Path:
    # PORT: the original scripts used different names (classifier: ...Seizure{j}Cross{c}.txt,
    # post-processing: ...Seizure{j}Win256Cross{c}.txt) and different folders; unified here.
    return PRED_DIR / f"Pat{patient}" / f"HDClassifier_Seizure{seizure}Cross{cross}.pt"

In [ ]:
%%writefile code/swec_ethz_short/LBP+HD/LBP+HD/iEEGFunctions.py
#!/usr/bin/env python
'''
Local sensitive Hashing function

Python 3 / PyTorch 2 port of the ETH Zurich iEEGFunctions.py (Burrello et al., BioCAS 2018).
Changes are marked "PORT:". Summary:
  * runs on CPU or any GPU (original: CUDA only, torch.cuda.*Tensor constructors)
  * deprecated torch.add(a, 1, b) calls rewritten as a + b
  * Python 3 integer division (fs/2 -> fs//2, HD_dim/2 -> HD_dim//2, (T-1)/2 -> (T-1)//2)
  * learn_HD_proj has a vectorised implementation that gives bit-identical results
    (set SWEC_FAST=0 to use the original sample-by-sample loop instead)
'''
import time, sys
import os
import math

import numpy as np
import scipy.special
import torch

__author__ = "Alessio Burrello"
__email__ = "alessio.burrello@studenti.polito.it"

# PORT: default device for the LBP/LGP feature functions used by the baselines
DEFAULT_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _as_device(device):
    # PORT: original passed a GPU index; accept an index, a string or a torch.device
    if isinstance(device, torch.device):
        return device
    if isinstance(device, int):
        return torch.device(f"cuda:{device}") if torch.cuda.is_available() else torch.device("cpu")
    return torch.device(device)


class HD_classifier:

    def __init__(self, N_seats1, HD_dim, N_seats2, T, device, string, cuda=True):
        ''' Initialize an HD classifier using the torch library
        Parameters
        ----------
        N_seats1: number of elements inside the LBP item memory
        HD_dim: dimension of the HD vectprs
        N_seats2: number of channels of the channels item memory
        T: LBP length + 1
        device: device used to create the itemMemory (PORT: GPU index, "cpu" or torch.device)
        string: type of item Memory for the channel item Memory: random or sandwich
        cuda: PORT: ignored, the device decides.
        '''
        self.training_done = False
        self.N_seats1 = N_seats1
        self.N_seats2 = N_seats2
        self.HD_dim = HD_dim
        self.T = T
        self.device = _as_device(device)
        # creation of a random itemMemory of 1 and 0 of dimension N_seats1xHD_dim
        # PORT: random numbers are drawn on the CPU and then moved, exactly like the
        # original torch.randn(...).cuda(), so the same seed gives the same memory on CPU and GPU.
        self.proj_mat_LBP = torch.randn(self.HD_dim, self.N_seats1).to(self.device)
        self.proj_mat_LBP[self.proj_mat_LBP > 0] = 1
        self.proj_mat_LBP[self.proj_mat_LBP <= 0] = 0
        # creation of a sandwich itemMemory of 1 and 0 of dimension N_seats2xHD_dim: the single
        # hypervectors inside the memory are similar to adiacent ones (~75% equal) and pseudo orthogonal
        # with the others.
        if string == 'sandwich':
            self.proj_mat_channels = torch.zeros(self.HD_dim, self.N_seats2, device=self.device)
            for i in range(N_seats2):
                if i % 2 == 0:
                    self.proj_mat_channels[:, i] = torch.randn(self.HD_dim, 1).view(-1).to(self.device)
                    self.proj_mat_channels[self.proj_mat_channels > 0] = 1
                    self.proj_mat_channels[self.proj_mat_channels <= 0] = 0
            h = HD_dim // 2   # PORT: HD_dim/2 -> HD_dim//2
            for i in range(N_seats2 - 1):
                if i % 2 == 1:
                    self.proj_mat_channels[0:h, i] = self.proj_mat_channels[0:h, i - 1]
                    self.proj_mat_channels[h:HD_dim, i] = self.proj_mat_channels[h:HD_dim, i + 1]
            self.proj_mat_channels[0:h, N_seats2 - 1] = self.proj_mat_channels[0:h, N_seats2 - 2]
            self.proj_mat_channels[h:HD_dim, N_seats2 - 1] = torch.randn(self.HD_dim // 2, 1).view(-1).to(self.device)
            self.proj_mat_channels[self.proj_mat_channels > 0] = 1
            self.proj_mat_channels[self.proj_mat_channels <= 0] = 0
        # creation of a random itemMemory of 1 and 0 of dimension N_seats2xHD_dim
        elif string == 'random':
            self.proj_mat_channels = torch.randn(self.HD_dim, self.N_seats2).to(self.device)
            self.proj_mat_channels[self.proj_mat_channels >= 0] = 1
            self.proj_mat_channels[self.proj_mat_channels < 0] = 0

        # PORT: helpers for the vectorised encoder (same values, compact dtypes)
        self.LBP_weights = torch.tensor([2**0, 2**1, 2**2, 2**3, 2**4, 2**5], dtype=torch.int16, device=self.device)
        self._lbp_u8 = self.proj_mat_LBP.to(torch.uint8)
        self._ch_u8 = self.proj_mat_channels.to(torch.uint8)
        self._tie = self.xor(self.proj_mat_LBP[:, 1], self.proj_mat_LBP[:, 2]).to(torch.int32)
        self.fast = os.environ.get("SWEC_FAST", "1") != "0"
        self.chunk = int(os.environ.get("SWEC_CHUNK", "32"))

    def learn_HD_proj(self, EEG):
        ''' starting from EEG data, it creates the histogram of LBP for the target segment
        projecting it in the HD space.
        Parameters
        ----------
        EEG: samples of EEG on multiple channels.
        Return
        ------
        quueryVector: HV in which all the histograms among all the channels are encoded.
        '''
        if self.fast and EEG.size(1) - 6 > 0:
            return self._learn_HD_proj_fast(EEG)
        return self.learn_HD_proj_reference(EEG)

    def learn_HD_proj_reference(self, EEG):
        ''' PORT: the original sample-by-sample algorithm, only API-modernised. '''
        queeryVector = torch.zeros(1, 1, dtype=torch.int16, device=self.device)
        N_channels, learningEnd = EEG.size()
        LBP_weights = self.LBP_weights
        for iStep in range(learningEnd - 6):
            x = EEG[:, iStep:(iStep + self.T)].short()
            bp = ((x[:, 1:self.T] - x[:, 0:self.T - 1]) > 0).short()   # PORT: was torch.add(-x[..], 1, x[..])
            value = torch.sum(torch.mul(LBP_weights, bp), dim=1)
            bindingVector = self.xor(self.proj_mat_channels, self.proj_mat_LBP[:, value.long()])
            output_vector = torch.sum(bindingVector, dim=1)
            # here we broke ties summing an additional HV in case of an even number
            if N_channels % 2 == 0:
                output_vector = self.xor(self.proj_mat_LBP[:, 1], self.proj_mat_LBP[:, 2]) + output_vector
            output_vector = (output_vector > int(math.floor(self.N_seats2 / 2))).short()
            queeryVector = queeryVector + output_vector
        queeryVector = (queeryVector > (learningEnd - 6) // 2).short()   # PORT: py2 int division
        return queeryVector

    def _learn_HD_proj_fast(self, EEG):
        ''' PORT: vectorised version of learn_HD_proj_reference (bit-identical output).
        Computes all LBP codes at once, then encodes `chunk` time steps per tensor operation.
        '''
        N_channels, learningEnd = EEG.size()
        n_steps = learningEnd - 6
        xs = EEG[:, :n_steps + self.T - 1].short()
        d = (xs[:, 1:] - xs[:, :-1] > 0).to(torch.int64)                 # (C, n_steps + T - 2)
        codes = torch.zeros(N_channels, n_steps, dtype=torch.int64, device=self.device)
        for k in range(self.T - 1):
            codes += (2 ** k) * d[:, k:k + n_steps]                       # LBP code of every step
        thr = int(math.floor(self.N_seats2 / 2))
        acc = torch.zeros(self.HD_dim, dtype=torch.int32, device=self.device)
        ch = self._ch_u8.unsqueeze(2)                                     # (D, C, 1)
        for s0 in range(0, n_steps, self.chunk):
            c = codes[:, s0:s0 + self.chunk]                              # (C, S)
            bound = self._lbp_u8[:, c] != ch                              # xor -> (D, C, S)
            out = bound.sum(dim=1, dtype=torch.int32)                     # (D, S)
            if N_channels % 2 == 0:
                out = out + self._tie.unsqueeze(1)
            acc += (out > thr).sum(dim=1, dtype=torch.int32)
        return (acc > n_steps // 2).short().unsqueeze(0)                  # (1, D) like the original

    def learn_HD_proj_big_half(self, EEG, fs):
        ''' Function used during training to take a bigger window of EEG and sum all the histograms
        of LBP frequency of 0.5 seconds windows inside.
        Parameters
        ----------
        EEG: samples of EEG on multiple channels.
        fs: frequency of the signal.
        Return
        ------
        quueryVector: HV in which all the informations of this big window of EEG is encoded.
        '''
        queeryVector = torch.zeros(1, 1, dtype=torch.int16, device=self.device)
        N_channels, learningEnd = EEG.size()
        half = fs // 2   # PORT: fs/2 -> fs//2
        index = np.arange(0, learningEnd, half)
        for iStep in index:
            temp = self.learn_HD_proj(EEG[:, iStep:iStep + half])
            queeryVector = queeryVector + temp
        queeryVector = (queeryVector > index.size // 2).short()
        return queeryVector

    def predict(self, testVector, Ictalprot, Interictalprot, D):
        ''' Prediction function of HD: it gives in addition to the class prediction also the
        distance from the 2 class prototypes.
        Parameters
        ----------
        testVector: HV of the unlabled segment.
        Ictalprot: prototype (HD vector) for the ictal state.
        Interictalprot: prototype (HD vector) for the interictal state.
        Return
        ------
        distanceVectorsS: distance from ictal Prototype
        distanceVectornS: distance from interictal prototype
        prediction: 1 for seizure, 0 for interictal.
        '''
        distanceVectorsS = self.ham_dist(testVector, Ictalprot, D)
        distanceVectorsnS = self.ham_dist(testVector, Interictalprot, D)
        if distanceVectorsS > distanceVectorsnS:
            prediction = 0
        else:
            prediction = 1
        return distanceVectorsS, distanceVectorsnS, prediction

    def xor(self, vec_a, vec_b):
        ''' xor between vec_a and vec_b
        Parameters
        ----------
        vec_a: first vector, torch Short Tensor shape (HD_dim,)
        vec_b: second vector, torch Short Tensor shape (HD_dim,)
        Return
        ------
        vec_c: vec_a xor vec_b
        '''
        vec_c = (torch.add(vec_a, vec_b) == 1).short()  # xor
        return vec_c

    def ham_dist(self, vec_a, vec_b, D):
        ''' calculate relative hamming distance
        Parameters
        ----------
        vec_a: first vector, torch Short Tensor shape (HD_dim,)
        vec_b: second vector, torch Short Tensor shape (HD_dim,)
        Return
        ------
        rel_dist: relative hamming distance
        '''
        vec_c = self.xor(vec_a, vec_b)
        rel_dist = torch.sum(vec_c) / float(D)
        return rel_dist


# ------------------------------------------------------------------------------------------
# Feature extractors used by the LBP+SVM / LGP+MLP baselines (not used by LBP+HD).
# PORT: device-agnostic; torch.cuda.FloatTensor(...) -> torch.zeros(..., device=...).
# ------------------------------------------------------------------------------------------

def LBP_window_train(EEG, fs, totalNumberBP, label, T, device=None):
    ''' Function used during training to take a bigger window of EEG and sum all the histograms
    of LBP frequency of 0.5 seconds windows inside.
    Return: Vector for train of dimension numberOfWindowsx(2^(T-1)xnChannels).
    '''
    device = device or EEG.device
    N_channels, learningEnd = EEG.size()
    half = fs // 2
    index = np.arange(0, learningEnd, half)
    queeryVector = torch.zeros(index.size, totalNumberBP * N_channels + 1, device=device)
    i = 0
    for iStep in index:
        temp = LBP_extractor(EEG[:, iStep:iStep + half], totalNumberBP, label, T, device)
        queeryVector[i, :] = temp.view(-1)   # PORT: (n,1) -> (n,)
        i = i + 1
    return queeryVector


def LBP_extractor(EEG, totalNumberBP, label, T, device=None):
    ''' starting from EEG data, it creates the histogram of LBP for the target segment.
    Return: Features_vector: in position 0 the label, then the full histogram (features).
    '''
    device = device or EEG.device
    N_channels, learningEnd = EEG.size()
    Features_vector = torch.zeros(totalNumberBP * N_channels + 1, 1, device=device)
    LBP_weights = torch.tensor([2**0, 2**1, 2**2, 2**3, 2**4, 2**5], dtype=torch.float32, device=device)
    chan_offset = torch.arange(N_channels, device=device, dtype=torch.float32) * totalNumberBP
    for iStep in range(learningEnd - 6):
        x = EEG[:, iStep:(iStep + T)].float()
        bp = ((x[:, 1:T] - x[:, 0:T - 1]) > 0).float()
        value = torch.sum(torch.mul(LBP_weights, bp), dim=1) + 1
        index = chan_offset + value
        Features_vector[index.long()] = Features_vector[index.long()] + 1
    Features_vector[0] = label
    return Features_vector


def LGP_window_train(EEG, fs, totalNumberBP, label, T, device=None):
    ''' Same as LBP_window_train but with local gradient patterns (LGP). '''
    device = device or EEG.device
    N_channels, learningEnd = EEG.size()
    half = fs // 2
    index = np.arange(0, learningEnd, half)
    queeryVector = torch.zeros(index.size, totalNumberBP * N_channels + 1, device=device)
    i = 0
    for iStep in index:
        temp = LGP_extractor(EEG[:, iStep:iStep + half], totalNumberBP, label, T, device)
        queeryVector[i, :] = temp.view(-1)
        i = i + 1
    return queeryVector


def LGP_extractor(EEG, totalNumberGP, label, T, device=None):
    ''' starting from EEG data, it creates the histogram of LGP for the target segment. '''
    device = device or EEG.device
    N_channels, learningEnd = EEG.size()
    Features_vector = torch.zeros(totalNumberGP * N_channels + 1, 1, device=device)
    LBP_weights = torch.tensor([2**0, 2**1, 2**2, 2**3, 2**4, 2**5], dtype=torch.float32, device=device)
    chan_offset = torch.arange(N_channels, device=device, dtype=torch.float32) * totalNumberGP
    c = (T - 1) // 2                         # PORT: (T-1)/2 -> (T-1)//2
    neigh = [0, 1, 2, 4, 5, 6]
    for iStep in range(4, learningEnd - 4):
        x = EEG[:, iStep - c:iStep + c + 1].float()
        x = torch.abs(x - x[:, c:c + 1])                                   # |x - centre sample|
        bp = ((x - torch.mean(x[:, neigh], 1, keepdim=True)) > 0).float()  # vs mean of neighbours
        value = torch.sum(torch.mul(LBP_weights, bp[:, neigh]), dim=1) + 1
        index = chan_offset + value
        Features_vector[index.long()] = Features_vector[index.long()] + 1
    Features_vector[0] = label
    return Features_vector

In [ ]:
%%writefile code/swec_ethz_short/LBP+HD/LBP+HD/iEEGClassifierLBP+HD.py
#!/usr/bin/env python
"""
iEEGClassifierLBP+HD.py - Python 3 / PyTorch 2 port (Burrello et al., BioCAS 2018)

Trains the LBP+HD classifier on 1..n seizures and tests it on every seizure of the
patient, for each cross-validation fold. One prediction file per (seizure, fold) is
written to  results\\swec_short\\LBP+HD\\predictions\\Pat<n>\\ .

Run (PowerShell, .venv active, from this folder):
    $env:SWEC_PATIENT = 6
    python "iEEGClassifierLBP+HD.py"
then run HD_PostProcessing.py with the same SWEC_PATIENT.
Other settings: see swec_config.py.

Changes vs. the original are marked "PORT:".
"""
import sys
import os
import time

import numpy as np
import scipy.io as scp
import torch

from iEEGFunctions import HD_classifier   # here there are all the functions that we use
import swec_config as cfg                 # PORT: shared paths/tables/device for both scripts

__author__ = "Alessio Burrello"
__email__ = "alessio.burrello@studenti.polito.it"

# PORT: was  device = 1; torch.cuda.set_device(device)   (second GPU of the ETH server)
device = cfg.get_device()
# Patient 9, the third seizure is an artifact!
Seizure_train = cfg.Seizure_train
seizureN = cfg.seizureN
fs = cfg.FS
# table that represents all the ictal segments used to train with each different
# patient and seizure (seconds from the start of each recording)
Seizure_begin = [[184, 184, 183, 186, 186, 0,   0,   0,   0,   0, 0, 0, 0, 0],
                 [180, 192, 188, 186, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
                 [181, 207, 231, 190, 190, 196, 193, 202, 205, 201, 195, 195, 194, 191],
                 [220, 219, 200, 192, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
                 [187, 188, 208, 202, 202, 212,   0,   0,   0,   0, 0, 0, 0, 0],
                 [184, 186, 0, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
                 [200, 210, 200, 200, 220, 200, 200,   0,   0,   0, 0, 0, 0, 0],
                 [185, 188, 185, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
                 [188, 204, 204, 214, 183, 203, 185,   0,   0,   0, 0, 0, 0, 0],
                 [181, 185, 182, 182, 181, 182,   180,   184, 181, 180, 181, 182, 180, 0],
                 [181, 180, 0, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
                 [210, 210, 200, 220, 200, 200, 210, 200, 200, 190, 0, 0, 0, 0],
                 [185, 185, 0, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
                 [195, 185, 194, 195, 184, 185, 235, 185, 182, 185, 0, 0, 0, 0],
                 [213, 208, 200, 204, 210, 200, 216, 205, 203,   0, 0, 0, 0, 0],
                 [205, 195, 0, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0]]
Seizure_end = [[192, 193, 189, 190, 200, 0,   0,   0,   0,   0, 0, 0, 0, 0],
               [210, 215, 215, 215, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
               [200, 230, 255, 220, 219, 219, 215, 228, 228, 226, 220, 213, 218, 210],
               [250, 255, 239, 217, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
               [225, 218, 234, 231, 221, 242,   0,   0,   0,   0, 0, 0, 0, 0],
               [193, 195, 0, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
               [220, 230, 220, 220, 240, 225, 220,   0,   0,   0, 0, 0, 0, 0],
               [205, 205, 210, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
               [201, 231, 210, 244, 195, 226, 211,   0,   0,   0, 0, 0, 0, 0],
               [189, 190, 196, 190, 183, 188,   181,   193, 190, 189, 191, 199, 181, 0],
               [211, 210, 0, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
               [225, 225, 215, 230, 220, 220, 220, 220, 220,   0, 0, 0, 0, 0],
               [210, 210, 0, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0],
               [210, 200, 218, 219, 214, 220, 268, 220, 222, 222, 0, 0, 0, 0],
               [238, 225, 225, 234, 235, 225, 241, 225, 233,   0, 0, 0, 0, 0],
               [235, 225, 0, 0, 0, 0,   0,   0,   0,   0, 0, 0, 0, 0]]
second = fs
minutes = second * 60
T = 7        # 1 + dimension l of binary pattern
totalNumberBP = 2 ** (T - 1)
D = 10000    # dimension of hypervector
half = fs // 2   # PORT: fs/2 -> fs//2 (0.5 s window = 256 samples)
# List of patient that we want to test
PatientList = cfg.patients_from_env(default=[2])   # PORT: was PatientList = [2]
# PORT: the original training loop read file Sz{i+1+cross} but its ictal times from
# column i, i.e. the wrong seizure whenever cross > 0. Fixed to column i+cross.
# Set SWEC_ORIGINAL_TRAIN_INDEX=1 to reproduce the original behaviour exactly.
ORIGINAL_TRAIN_INDEX = os.environ.get("SWEC_ORIGINAL_TRAIN_INDEX", "0") == "1"


def load_eeg(path):
    """EEG matrix as stored: samples x channels."""
    return np.asarray(scp.loadmat(str(path), variable_names=["EEG"])["EEG"])


print(f"[run] device={device}  seed={cfg.seed()}  folds={os.environ.get('SWEC_FOLDS', 'full')}  "
      f"original_train_index={ORIGINAL_TRAIN_INDEX}  fast_encoder={os.environ.get('SWEC_FAST', '1') != '0'}")

for Patient in PatientList:
    K = cfg.folds(Patient)
    # PORT: the original read this first file from Pat{Patient_trasformation[Patient-1]} and all
    # others from Pat{Patient}; one consistent folder is used here (see swec_config.mapping()).
    first = load_eeg(cfg.seizure_file(Patient, 1))
    nSignals = first.shape[1]
    del first
    print(f"\n=== Patient {Patient} -> {cfg.data_folder(Patient).name}: {nSignals} channels, "
          f"{seizureN[Patient-1]} seizures, train on {Seizure_train[Patient-1]}, {K} fold(s) ===")
    torch.manual_seed(cfg.seed())
    # creating the model with the 2 itemMemory, both random and binarized
    model = HD_classifier(totalNumberBP, D, nSignals, T, device, 'random')
    t_patient = time.time()
    for cross in range(K):
        ####TRAINING####
        for i in range(Seizure_train[Patient-1]):
            sz = i + 1 + cross
            col = i if ORIGINAL_TRAIN_INDEX else i + cross
            eeg = load_eeg(cfg.seizure_file(Patient, sz))
            seg = eeg[Seizure_begin[Patient-1][col]*fs:Seizure_end[Patient-1][col]*fs, :]
            EEGSez = torch.from_numpy(np.ascontiguousarray(seg)).to(device).t()
            print(f'Learning Seizure number {sz}  ({Seizure_begin[Patient-1][col]}-{Seizure_end[Patient-1][col]} s)')
            temp1 = model.learn_HD_proj_big_half(EEGSez, fs)
            # I use the first seizure to create the HV: if we have more than 1, I will sum the other
            # to the first HV and perform an other binarization of the HV.
            if i == 0:
                queeryVectorS0 = temp1
            else:
                queeryVectorS0 = queeryVectorS0 + temp1
        queeryVectorS0 = (queeryVectorS0 > Seizure_train[Patient-1] // 2).short()
        eeg = load_eeg(cfg.seizure_file(Patient, 1))
        EEGInterictal = torch.from_numpy(np.ascontiguousarray(eeg[10*fs:50*fs, :])).to(device).t()
        print('Learning Interictal period')
        temp3 = model.learn_HD_proj_big_half(EEGInterictal, fs)
        queeryVectornS0 = temp3
        ####TESTING####
        t = time.time()
        for j in range(1, seizureN[Patient-1] + 1):
            eeg = load_eeg(cfg.seizure_file(Patient, j))
            EEGtest = torch.from_numpy(eeg).to(device).t()
            index = np.arange(0, EEGtest.size(1), half)
            # PORT: was torch.zeros(1, f['EEG'].shape[0]/fs*2) (float size in Python 3)
            distanceVectorsS0 = torch.zeros(1, len(index), device=device)
            distanceVectorsnS0 = torch.zeros(1, len(index), device=device)
            prediction0 = torch.zeros(1, len(index), device=device)
            t_sz = time.time()
            for i, iStep in enumerate(index):
                temp = model.learn_HD_proj(EEGtest[:, iStep:iStep + half])
                [distanceVectorsS0[0, i], distanceVectorsnS0[0, i], prediction0[0, i]] = \
                    model.predict(temp, queeryVectorS0, queeryVectornS0, D)
            # PORT: the original printed every half second; one line per seizure instead
            print(f'Patient {Patient} fold {cross+1}/{K} seizure {j}/{seizureN[Patient-1]}: '
                  f'{len(index)} windows, {int(prediction0.sum().item())} predicted ictal, {time.time()-t_sz:.1f} s')
            # for the purpose of this code, we save only prediction, discarding the 2 distances from the 2 prototypes
            out = cfg.prediction_file(Patient, j, cross)
            out.parent.mkdir(parents=True, exist_ok=True)
            torch.save(prediction0[0, :].cpu(), out)
    print(f'Patient {Patient} finished in {(time.time()-t_patient)/60:.1f} min. '
          f'Predictions in {cfg.PRED_DIR / ("Pat" + str(Patient))}')
# PORT: all paths now come from swec_config.py (the original pointed to the ETH network).

In [ ]:
%%writefile code/swec_ethz_short/LBP+HD/LBP+HD/HD_PostProcessing.py
"""
HD_PostProcessing.py - Python 3 / PyTorch 2 port (Burrello et al., BioCAS 2018)

Applies the moving average (10 half-second windows) and the patient threshold to the
predictions written by iEEGClassifierLBP+HD.py, and prints specificity, sensitivity
and mean detection delay. Also appends one row per patient to
results\\swec_short\\LBP+HD\\summary.csv.

Run with the SAME settings as the classifier:
    $env:SWEC_PATIENT = 6
    python HD_PostProcessing.py

Changes vs. the original are marked "PORT:".
"""
import csv
import os
from datetime import datetime

import torch
import numpy

import swec_config as cfg   # PORT: shared paths/tables/device

# we can run the postProcessing for 1 Patient at the time.
dim = 5
Threshold_size = 10
fs = cfg.FS
device = cfg.get_device()             # PORT: was torch.cuda.set_device(1)
seizureN = cfg.seizureN               # number of seizures of each patient
seizureTrain = cfg.Seizure_train      # number of seizures used for training
threshold = cfg.threshold             # threshold of postprocessing
win_dim = int(numpy.ceil(fs * dim / 10))   # PORT: numpy.int was removed from NumPy
PatientList = cfg.patients_from_env(default=[2])   # PORT: was a commented list with only 14 active

summary_csv = cfg.RESULTS_DIR / "summary.csv"

for Patient in PatientList:
    K = cfg.folds(Patient)            # PORT: same folds as the classifier (see swec_config.py)
    number_seizures = list(range(seizureN[Patient-1]))
    cross = list(range(K))
    counter1 = 0
    l = 0
    delay = 0
    print(f'=== Patient {Patient}: {seizureN[Patient-1]} seizures, {K} fold(s), threshold {threshold[Patient-1]} ===')
    for j in cross:
        for i in number_seizures:
            # loading each file created with the test of a single seizure using each other seizure to train
            # Example with 2 seizure: we will have for example 2 different files for seizure 1, one trained with seizure 1, one with seizure 2
            # PORT: path/name unified with the classifier (see cfg.prediction_file)
            prediction = torch.load(cfg.prediction_file(Patient, i + 1, j), map_location=device)
            Pred_mean = torch.zeros(prediction.size()[0], device=device)
            # Threhsolding the results inside the big 5 second window
            for k in numpy.arange(Threshold_size, prediction.size()[0]):
                Pred_mean[k] = torch.mean(prediction[k-Threshold_size:k])
            Pred_mean = torch.add(Pred_mean, threshold[Patient-1])
            Final_prediction = torch.sign(Pred_mean)
            First = 1
            counter = 0
            # updating of the delay and seizure detection (yes or not) for that particular recording
            for k in numpy.arange(0, Final_prediction.size()[0]):
                if Final_prediction[k] == 1 and First == 1 and k > 170*10/dim:
                    print(k/(10/float(dim)))
                    l = l + 1
                    delay = k/(10/float(dim)) - 180 + delay
                    First = 0
                elif Final_prediction[k] == 1 and k < 170*10/dim:
                    counter = counter + 1
            counter1 = counter1 + counter
            print('number of false:' + str(counter))
        print('end fold')
    # statistics computation and printing
    stamp = (1 - counter1/float(170*10/dim*seizureN[Patient-1]*K))*100
    stamp2 = (l) / float(seizureN[Patient-1]*K)*100
    print('Specificity =' + str(stamp))
    print('Sensitivity =' + str(stamp2))
    print('-----')
    # raw delay estimation, considering also the seizures used for train
    delay = delay/l if l else float('nan')   # PORT: avoid division by zero when nothing is detected
    print('Mean delay: ' + str(delay))

    # PORT: keep a machine-readable record for the replication table
    summary_csv.parent.mkdir(parents=True, exist_ok=True)
    new = not summary_csv.exists()
    with open(summary_csv, "a", newline="") as fh:
        w = csv.writer(fh)
        if new:
            w.writerow(["timestamp", "patient", "data_folder", "folds", "fold_mode", "seed",
                        "original_train_index", "threshold", "specificity", "sensitivity", "mean_delay_s"])
        w.writerow([datetime.now().isoformat(timespec="seconds"), Patient, cfg.data_folder(Patient).name, K,
                    os.environ.get("SWEC_FOLDS", "full"), cfg.seed(),
                    os.environ.get("SWEC_ORIGINAL_TRAIN_INDEX", "0"), threshold[Patient-1],
                    round(stamp, 2), round(stamp2, 2), round(delay, 2)])
    print(f'Row appended to {summary_csv}')

In [ ]:
%%writefile code/swec_ethz_short/LBP+HD/LBP+HD/paper_protocol_metrics.py
#!/usr/bin/env python
"""
paper_protocol_metrics.py  (NEW - put next to HD_PostProcessing.py)

Recomputes LBP+HD metrics from the saved prediction files under two protocols:
  released : what HD_PostProcessing.py reports - every recording in every fold that was run
             (SWEC_FOLDS), including the seizures used for training
  paper    : BioCAS 2018 protocol - unseen seizures only;
             Table I patients: k-fold;  Table II patients (3,7,9,10,12,14): fold 0 only
             (train on the first m seizures, test on the remaining ones)
Works with SWEC_FOLDS=full (lab run) and SWEC_FOLDS=original (paper protocol only).
Patients without prediction files are skipped.
Same moving-average/threshold/delay logic as HD_PostProcessing.py. No classifier re-run needed.

Usage:  python paper_protocol_metrics.py
Output: results\\swec_short\\LBP+HD\\LBPHD_replication_vs_BioCAS2018.csv
"""
import csv
import numpy as np
import torch
import swec_config as cfg

dim, TS = 5, 10
PRE = 170 * 10 / dim                      # first 170 s = 340 half-second windows
TABLE2 = {3, 7, 9, 10, 12, 14}
PAPER = {1: (6.3, 100, 100), 2: (15.1, 100, 100), 4: (34.5, 100, 100), 5: (20.9, 100, 100), 6: (6.3, 100, 100),
         8: (13.2, 100, 100), 11: (7.0, 100, 100), 13: (10.0, 100, 100), 15: (36.4, 100, 100), 16: (32.3, 100, 100),
         3: (21.8, 94.17, 100), 7: (5.0, 44, 100), 9: (16.2, 100, 100), 10: (9.3, 99.47, 90),
         12: (12.6, 93.10, 100), 14: (21.7, 100, 100)}   # (mean delay s, spe %, sen %) from Tables I-II
ORDER = [1, 2, 4, 5, 6, 8, 11, 13, 15, 16, 3, 7, 9, 10, 12, 14]


def analyse(p, folds, exclude_train):
    m = cfg.Seizure_train[p - 1]
    fp = n_rec = det = 0
    delays = []
    for c in folds:
        train = set(range(c + 1, c + m + 1))          # seizures trained on in fold c
        for s in range(1, cfg.seizureN[p - 1] + 1):
            if exclude_train and s in train:
                continue
            pred = torch.load(cfg.prediction_file(p, s, c), map_location="cpu")
            pm = torch.zeros(pred.numel())
            for k in range(TS, pred.numel()):
                pm[k] = torch.mean(pred[k - TS:k])
            fin = torch.sign(pm + cfg.threshold[p - 1])
            first = True
            n_rec += 1
            for k in range(fin.numel()):
                if fin[k] == 1 and first and k > PRE:
                    det += 1
                    delays.append(k / (10 / dim) - 180)
                    first = False
                elif fin[k] == 1 and k < PRE:
                    fp += 1
    return dict(n=n_rec, spe=round((1 - fp / (PRE * n_rec)) * 100, 2), sen=round(det / n_rec * 100, 2),
                delay=round(float(np.mean(delays)), 2) if delays else float("nan"))


out = cfg.RESULTS_DIR / "LBPHD_replication_vs_BioCAS2018.csv"
with open(out, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["paper_table", "patient", "dataset_folder", "test_recordings",
                "paper_delay_s", "paper_spe", "paper_sen", "repl_delay_s", "repl_spe", "repl_sen", "delay_diff_s",
                "released_delay_s", "released_spe", "released_sen"])
    for p in ORDER:
        run_folds = range(cfg.folds(p))
        if not cfg.prediction_file(p, cfg.seizureN[p - 1], run_folds[-1]).exists():
            print(f"P{p:<3} skipped - predictions missing (run the classifier for this patient)")
            continue
        rel = analyse(p, run_folds, exclude_train=False)
        pap = analyse(p, [0] if p in TABLE2 else run_folds, exclude_train=True)
        d, s, n = PAPER[p]
        row = ["II" if p in TABLE2 else "I", p, cfg.data_folder(p).name, pap["n"], d, s, n,
               pap["delay"], pap["spe"], pap["sen"], round(pap["delay"] - d, 2), rel["delay"], rel["spe"], rel["sen"]]
        w.writerow(row)
        print(f"P{p:<3} paper {d:>5}/{s:>6}/{n:>4}   replication {pap['delay']:>6}/{pap['spe']:>6}/{pap['sen']:>6}")
print(f"\nWritten to {out}")

In [ ]:
# Self-test 1: the fast encoder must give exactly the same hypervectors as the original algorithm
import importlib, numpy as np, torch
import iEEGFunctions, swec_config as cfg
importlib.reload(iEEGFunctions); importlib.reload(cfg)
from iEEGFunctions import HD_classifier

for C in (7, 8, 36):                         # odd and even channel counts
    torch.manual_seed(1)
    m = HD_classifier(64, 10000, C, 7, "cpu", "random")
    eeg = torch.from_numpy(np.random.default_rng(C).normal(0, 40, (C, 256)))
    same = torch.equal(m.learn_HD_proj_reference(eeg), m._learn_HD_proj_fast(eeg))
    print(f"{C:3d} channels: fast encoder identical to original -> {same}")
    assert same

# Self-test 2: patient -> data folder mapping (expected: 'transform', 16 patients checked)
print("mapping:", cfg.mapping(), "| paper patient 6 ->", cfg.data_folder(6).name)
if "git_commit" in globals():
    git_commit("LBP+HD ported to Python 3 / PyTorch 2 (+ verify and metrics scripts)")

## Step 8 — Train and test LBP+HD for every patient
For each patient this runs, as separate processes, exactly what was run in the lab PC terminal:
```
python "iEEGClassifierLBP+HD.py"   ->  results\swec_short\LBP+HD\predictions\Pat<n>\*.pt
python HD_PostProcessing.py        ->  specificity / sensitivity / mean delay  (+ row in summary.csv)
```
Output is shown here and saved in `results\lbphd_P<n>.log`.

**Run time on the lab PC (CPU only, `FOLD_MODE="full"`): about 2 days in total.** Patient 6 takes about 10 min, patient 10 about 10 h and patient 3 about 10 h. With a GPU it is much faster.
* **Keep the computer awake:** Windows Settings → System → Power → Sleep: *Never*.
* **Do not open `summary.csv` in Excel while this runs.** Excel locks the file. Step 9 rebuilds it anyway.
* **You can stop this cell at any time and run it again later.** Completed patients are skipped (`RERUN_EXISTING = False`).

### 8a — Smoke test: patient 6
This is the smallest patient (2 seizures, 2 folds, about 10 min on CPU). Expected result: specificity 100, sensitivity 100, mean delay about 6.25 s. The paper reports 6.3 s, 100 %, 100 %.

In [ ]:
import importlib, swec_config as cfg
importlib.reload(cfg)

def predictions_complete(p):
    K = cfg.folds(p)
    return all(cfg.prediction_file(p, s, c).exists() for c in range(K) for s in range(1, cfg.seizureN[p - 1] + 1))

def run_patient(p):
    log = RESULTS / f"lbphd_P{p}.log"
    env = {"SWEC_PATIENT": str(p)}
    t0 = time.time()
    if predictions_complete(p) and not RERUN_EXISTING:
        print(f"=== Patient {p}: predictions already complete - classifier skipped ===")
    else:
        print(f"=== Patient {p}: training + testing ({cfg.folds(p)} fold(s)) - started {time.strftime('%H:%M')} ===")
        rc = run([sys.executable, "iEEGClassifierLBP+HD.py"], cwd=LBPHD_DIR, log=log, env=env)
        if rc != 0:
            print(f"!!! classifier failed for patient {p} (exit {rc}) - see {log}")
            return False
    rc = run([sys.executable, "HD_PostProcessing.py"], cwd=LBPHD_DIR, log=log, append=True, env=env)
    print(f"=== Patient {p} done in {(time.time() - t0) / 60:.1f} min ===\n")
    return rc == 0

run_patient(6)

### 8b — All patients (long run)

In [ ]:
t_all = time.time()
failed = [p for p in RUN_PATIENTS if not run_patient(p)]
print(f"Finished in {(time.time() - t_all) / 3600:.1f} h. Failed patients: {failed or 'none'}")

## Step 9 — Rebuild `summary.csv` and compute the paper-protocol metrics
* **`summary.csv`** (released-code metrics): rebuilt from scratch, one row per patient. Duplicate rows from restarts in Step 8 disappear. The metrics average over every recording in every fold, *including* the seizures used for training. This is what `HD_PostProcessing.py` reports.
* **`LBPHD_replication_vs_BioCAS2018.csv`** (paper protocol): scores *unseen* seizures only. Table I patients use k-fold; Table II patients (3, 7, 9, 10, 12, 14) use a single split: train on the first *m* seizures and test on the rest. These are the numbers to compare with the paper.

Both are computed from the saved predictions; nothing is retrained.

In [ ]:
import importlib, swec_config as cfg
importlib.reload(cfg)
def predictions_complete(p):
    return all(cfg.prediction_file(p, s, c).exists()
               for c in range(cfg.folds(p)) for s in range(1, cfg.seizureN[p - 1] + 1))
done = [p for p in range(1, 17) if predictions_complete(p)]
print("Patients with complete predictions:", done)

summary = LBPHD_RES / "summary.csv"
try:
    summary.unlink(missing_ok=True)
except PermissionError:
    raise SystemExit("summary.csv is open in another program (Excel?) - close it and run this cell again.")

run([sys.executable, "HD_PostProcessing.py"], cwd=LBPHD_DIR, log=RESULTS / "postprocessing_all.log",
    env={"SWEC_PATIENT": ",".join(map(str, done))})
print()
run([sys.executable, "paper_protocol_metrics.py"], cwd=LBPHD_DIR, log=RESULTS / "paper_protocol_metrics.log")

## Step 10 — Compare with the paper (Tables I and II) and with the lab run

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
cmp = pd.read_csv(LBPHD_RES / "LBPHD_replication_vs_BioCAS2018.csv")

cols = ["patient", "dataset_folder", "test_recordings", "paper_delay_s", "repl_delay_s", "delay_diff_s",
        "paper_spe", "repl_spe", "paper_sen", "repl_sen"]
t1, t2 = cmp[cmp.paper_table == "I"], cmp[cmp.paper_table == "II"]
print("TABLE I  - learning from 1-2 seizures, k-fold (paper protocol)")
display(t1[cols].reset_index(drop=True))
print("TABLE II - learning from 3-6 seizures, single split (paper protocol)")
display(t2[cols].reset_index(drop=True))

# headline numbers
missed = int(round(((100 - cmp.repl_sen) / 100 * cmp.test_recordings).sum()))
print(f"Table I : mean delay {t1.repl_delay_s.mean():.1f} s (paper {t1.paper_delay_s.mean():.1f} s); "
      f"patients with 100 % spe and sen: {int(((t1.repl_spe == 100) & (t1.repl_sen == 100)).sum())}/{len(t1)}")
print(f"Table II: mean specificity {t2.repl_spe.mean():.1f} % (paper {t2.paper_spe.mean():.1f} %); "
      f"mean sensitivity {t2.repl_sen.mean():.1f} % (paper {t2.paper_sen.mean():.1f} %)")
print("Unseen-seizure tests missed:", missed, "(the lab run and the paper both miss 1 of the 65 unseen seizures, patient 10)")

In [ ]:
# Reproducibility check: same seed + same code should give the same numbers as the lab PC run (2026-10-03).
LAB = {
    1: (7.21, 100.0, 100.0),
    2: (16.04, 100.0, 100.0),
    3: (15.91, 87.19, 100.0),
    4: (34.04, 100.0, 100.0),
    5: (25.03, 100.0, 100.0),
    6: (6.25, 100.0, 100.0),
    7: (-9.38, 11.62, 100.0),
    8: (13.58, 100.0, 100.0),
    9: (15.67, 100.0, 100.0),
    10: (4.11, 98.32, 90.0),
    11: (7.25, 100.0, 100.0),
    12: (0.25, 71.4, 100.0),
    13: (10.25, 100.0, 100.0),
    14: (5.42, 97.6, 100.0),
    15: (36.04, 100.0, 100.0),
    16: (31.0, 100.0, 100.0),
}   # patient: (delay s, specificity %, sensitivity %) - paper protocol, lab PC
diff = []
for _, r in cmp.iterrows():
    d, s, n = LAB[int(r.patient)]
    if abs(r.repl_delay_s - d) > 0.01 or abs(r.repl_spe - s) > 0.01 or abs(r.repl_sen - n) > 0.01:
        diff.append((int(r.patient), (r.repl_delay_s, r.repl_spe, r.repl_sen), (d, s, n)))
if not diff and len(cmp) == 16:
    print("All 16 patients identical to the lab-PC run.")
else:
    print(f"{len(cmp)} patients compared; differences (patient, this run, lab run):", *diff, sep="\n  ")
    print("Small differences can appear with FOLD_MODE / SEED changes or GPU vs CPU arithmetic; large ones mean a setup problem.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PAPER_C, REPL_C = "#2a78d6", "#eb6834"          # validated categorical pair (blue = paper, orange = replication)
INK, MUTED, SURF = "#0b0b0b", "#52514e", "#fcfcfb"
plt.rcParams.update({"axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

def bars(ax, df, pcol, rcol, ylabel, title):
    x = np.arange(len(df)); w = 0.38
    ax.bar(x - w/2 - 0.01, df[pcol], w, color=PAPER_C, label="Paper (BioCAS 2018)")
    ax.bar(x + w/2 + 0.01, df[rcol], w, color=REPL_C, label="Replication")
    ax.set_xticks(x, [f"P{p}" for p in df.patient])
    ax.set_ylabel(ylabel); ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.grid(axis="y", color="#e6e5e1", linewidth=0.8); ax.set_axisbelow(True)
    ax.axhline(0, color=MUTED, linewidth=0.8)
    ax.set_facecolor(SURF)

fig, axes = plt.subplots(2, 1, figsize=(11, 7.5), facecolor=SURF, gridspec_kw={"height_ratios": [1.3, 1]})
bars(axes[0], cmp, "paper_delay_s", "repl_delay_s", "Mean detection delay (s)",
     "Mean detection delay per patient  -  Table I patients | Table II patients")
axes[0].axvline(len(t1) - 0.5, color=MUTED, linewidth=0.8, linestyle="--")
bars(axes[1], t2, "paper_spe", "repl_spe", "Specificity (%)", "Specificity, Table II patients (Table I: 100 % in both)")
axes[1].set_ylim(0, 105)
axes[0].legend(frameon=False, loc="lower right", ncol=2, bbox_to_anchor=(1, 1.0))   # one legend for both panels
fig.tight_layout()
out = PLOTS / "lbphd_vs_paper.png"
fig.savefig(out, dpi=150, facecolor=SURF)
plt.close(fig)
from IPython.display import Image, display
display(Image(filename=str(out)))
print("Saved", out)

### What the lab run showed (2026-10-03)
* **Table I (10 patients): fully reproduced.**
  * Specificity and sensitivity are 100 % for all 10 patients, as in the paper.
  * Mean delay is 18.7 s against the paper's 18.2 s. Nine of the 10 patients are within 1.3 s; patient 5 is 25.0 s against 20.9 s.
* **Table II (6 patients): sensitivity matches; specificity is lower for three patients.**
  * Sensitivity matches exactly, including patient 10's one missed seizure.
  * Specificity matches patient 9 and is close for patients 10 and 14.
  * It is lower for patient 3 (87.2 vs 94.2 %), patient 12 (71.4 vs 93.1 %) and patient 7 (11.6 vs 44 %).
  * Where specificity is lower, the detector fires early, so delays are shorter, sometimes negative (patient 7). That reflects false alarms, not faster detection.
* **Likely cause:** the random hypervectors. PyTorch 2 generates different vectors from the 2018 version even with the authors' seed, and the per-patient thresholds were tuned to the authors' vectors.
  * To measure the effect, re-run the Table II patients with other seeds: set `SEED`, `FOLD_MODE = "original"` and `RUN_PATIENTS = [3, 7, 9, 10, 12, 14]` in Step 1, then run Steps 8–10.
  * Each seed overwrites the predictions, so copy `results\swec_short\LBP+HD` aside between seeds.